# Chain Reaction — Jane Street, July 2014


[Puzzle page](https://www.janestreet.com/puzzles/chain-reaction-index/)

## Answer

$$77$$

## AI disclaimer

I asked Claude to write this one up as a **walkthrough** rather than a bare solution: all of the
code and most of the prose is Claude's. The point is not the number at the top — it is the route
from "this is secretly a graph problem" to a proved optimum, and then using the solver to explain
*why* the answer is 77 and not 78.

## 1. What kind of problem is this?

This is a graph longest path problem.  We construct a graph by created 100 vertices and connecting each vertice to any number it divides or can be divided by.  Then, our problem is to find the longest traversal that does not repeat a vertice. 

Thinking of it as a graph allows us to reason about the problem quite well to help prune the solution space.  


The plan:

1. Reason about the chain by considering the graphs structure to help establish an upper bound and tight constraints
2. Build a greedy chain — a lower bound
3. hand the exact problem to CP-SAT

In [2]:
from ortools.sat.python import cp_model

### Helper functions

Two numbers may sit next to each other exactly when one divides the other. That rule is the only
piece of the puzzle, so it gets written once, in one function, and everything else calls it.

In [1]:
LARGEST = 100
NUMBERS = list(range(1, LARGEST + 1))

EXAMPLE_CHAIN = [37, 74, 2, 8, 4, 16, 48, 6, 3, 9, 27, 81]


def divides_either_way(x, y):
    if y % x == 0:
        return True
    if x % y == 0:
        return True
    return False


def neighbours_of(x):
    neighbours = []
    for y in NUMBERS:
        if y != x and divides_either_way(x, y):
            neighbours.append(y)
    return neighbours


NEIGHBOURS = {
    x: neighbours_of(x) for x in NUMBERS
}  # dictionary for looking up a number's neighbours

EDGES = []
for x in NUMBERS:
    for y in NEIGHBOURS[x]:
        if x < y:
            EDGES.append((x, y))

print(len(NUMBERS), "numbers and", len(EDGES), "legal links between them")

100 numbers and 382 legal links between them


With onlly 382 edges, this problem could be a lot worse!

### Structure

We can make some observations about the graph.

1. Primes over 50 (53, 59, 61, 67, 71, 73, 79, 83, 89, and 97) can only sit next to 1.  This means that at most 1 odd prime over 50 can be on our chain, at the beginning or end.  We lose 9 numbers
2. Primes over 33 are restricted to only 2 neighbors (1 and double itself)
3. Numbers over 50 must be non-adjacent.  I.E. we can never have two of them in a row.
4. Odd Numbers over 50 must be divisible by odd numbers under 33.  This means the set [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33] has to handle that work load.
3. Odd numbers over 50 are also very restricted -- can only sit between two odd numbers that are factors of itself and 1.  


In [3]:
def show_least_connected(count):
    """Print the `count` numbers with the fewest legal neighbours, and who those neighbours are."""
    by_degree = sorted(NUMBERS, key=lambda x: len(NEIGHBOURS[x]))
    for x in by_degree[:count]:
        print(f"{x:>3}  degree {len(NEIGHBOURS[x]):>2}   {NEIGHBOURS[x]}")


show_least_connected(20)

 53  degree  1   [1]
 59  degree  1   [1]
 61  degree  1   [1]
 67  degree  1   [1]
 71  degree  1   [1]
 73  degree  1   [1]
 79  degree  1   [1]
 83  degree  1   [1]
 89  degree  1   [1]
 97  degree  1   [1]
 37  degree  2   [1, 74]
 41  degree  2   [1, 82]
 43  degree  2   [1, 86]
 47  degree  2   [1, 94]
 29  degree  3   [1, 58, 87]
 31  degree  3   [1, 62, 93]
 49  degree  3   [1, 7, 98]
 51  degree  3   [1, 3, 17]
 55  degree  3   [1, 5, 11]
 57  degree  3   [1, 3, 19]


## 3. Checker

We will test it on the example chain.

In [8]:
def check(chain):
    for value in chain:
        if value < 1 or value > LARGEST:
            return False

    already_seen = set()
    for value in chain:
        if value in already_seen:
            return False
        already_seen.add(value)

    for position in range(len(chain) - 1):
        x = chain[position]
        y = chain[position + 1]
        if not divides_either_way(x, y):
            return False

    return True

In [9]:
print("the puzzle's example :", str(check(EXAMPLE_CHAIN)))
print("a repeated number    :", str(check([2, 4, 2])))
print("a broken link        :", str(check([2, 4, 6])))
print("out of range         :", str(check([2, 200])))

the puzzle's example : True
a repeated number    : False
a broken link        : False
out of range         : False


## 5. Upper bounds by hand

This is where the puzzle actually lives. Four observations, each of which you can verify by
staring at the degree table above.

### 5.1 Ten numbers are dead ends

Take a prime $$p$$ with $$50 < p \le 100$$. Its only divisors are $$1$$ and $$p$$, and its smallest
multiple $$2p$$ is over 100. So it touches exactly one other dot in the whole graph: **1**.

There are ten such primes: 53, 59, 61, 67, 71, 73, 79, 83, 89, 97.

A dot with a single neighbour must be an **end** of the path, and a path has two ends. Worse, both
ends can't be dead-end primes at once — that would make the whole chain $$p, 1, q$$, three numbers
long, because 1 can only be visited once. So **at most one** of the ten can appear in any chain
worth having, and at least nine numbers are lost before we start:

$$\text{longest chain} \le 91.$$

### 5.2 Nothing above 50 divides anything else in range

If $$50 < x < y \le 100$$ and $$x$$ divides $$y$$, then $$y \ge 2x > 100$$ — impossible. So the 50
numbers from 51 to 100 are **pairwise non-adjacent**: an independent set, half the graph, with no
internal links at all.

In a path, no two of them can be neighbours. So if a chain uses $$b$$ numbers above 50 and $$s$$
numbers below, it has to alternate, giving $$b \le s + 1$$. The top half of the range is 50 leaves
dangling off the bottom half, and every one of them has to be reached and left again through a
small number. **The bottom half does all the connecting work**, and its capacity is what the answer
really turns on.

In [ ]:
def count_links_above(threshold):
    """How many legal links join two numbers that are both greater than `threshold`."""
    count = 0
    for x, y in EDGES:
        if x > threshold and y > threshold:
            count = count + 1
    return count


print("links between two numbers over 50:", count_links_above(50))

### 5.3 The odd numbers over 50 are stranded, and this is the real bottleneck

Sharpen §5.2. Let $$y > 50$$ be **odd**. A neighbour of $$y$$ is either a multiple (over 100, so
none exist) or a divisor. Every divisor of an odd number is odd, and a proper divisor of $$y$$ is at
most $$y/3 < 34$$. An even number can never divide an odd one.

So: **every neighbour of an odd number over 50 is an odd number below 34.** The odd top half of the
range hangs entirely off a handful of small odd hubs — 3, 5, 7, 11, 13, 17, 19 — plus 1. And in a
path, *each of those hubs has only two slots*.

That is a counting argument waiting to happen. Let

$$H = \{1, 3, 5, 7, 11, 13, 17, 19\}$$

and let $$T$$ be the numbers all of whose neighbours lie in $$H$$. Every member of $$T$$ that the
chain uses consumes two hub slots (one if it happens to be an end of the chain), and there are only
$$8 \times 2 = 16$$ slots in total. With at most two chain ends,

$$2t - 2 \le 16 \quad\Longrightarrow\quad t \le 9.$$

And that is *before* the hubs spend any slots on the rest of the chain — which they must, or the
chain consists of nothing but hubs and $$T$$. The true limit is lower; §8 will pin it down.

In [ ]:
HUBS = [1, 3, 5, 7, 11, 13, 17, 19]


def every_neighbour_is_a_hub(x):
    """True when x can only ever be linked to a number in HUBS."""
    for y in NEIGHBOURS[x]:
        if y not in HUBS:
            return False
    return True


HUB_FED_NUMBERS = []
for x in NUMBERS:
    if x not in HUBS and every_neighbour_is_a_hub(x):
        HUB_FED_NUMBERS.append(x)

print("numbers fed only by the hubs:", HUB_FED_NUMBERS)
print(
    "that is", len(HUB_FED_NUMBERS), "numbers competing for", 2 * len(HUBS), "hub slots"
)

### 5.4 Four two-step tails hanging off the number 2

The primes 37, 41, 43, 47 sit just below the halfway line, so each has exactly one multiple in
range. Their neighbourhoods are tiny:

- $$37$$ links only to $$\{1, 74\}$$, and $$74$$ links only to $$\{1, 2, 37\}$$;
- likewise for 41/82, 43/86, 47/94.

Each pair is a **two-dot tail dangling off the number 2**. To include $$p$$ at all, either $$p$$ is
an end of the chain, or $$p$$ sits between its two neighbours $$1$$ and $$2p$$. There are only two
ends and only one number 1, so at most three tails can be used — and §8 will show that even two is
already a bad trade.

In [ ]:
TAIL_PRIMES = [37, 41, 43, 47]
for p in TAIL_PRIMES:
    print(f"{p:>3} -> {NEIGHBOURS[p]}     {2 * p:>3} -> {NEIGHBOURS[2 * p]}")

### 5.5 Where hand reasoning leaves us

We have a floor of 58 and a ceiling of 91, plus a strong suspicion that the ceiling is soft:
§5.1, §5.3 and §5.4 are all competing for the *same* two chain ends and the *same* number 1.
Turning that competition into an exact number by hand is genuinely painful. That is exactly the
kind of bookkeeping a solver is for.

## 6. The model: longest path via `add_circuit`

CP-SAT has no "longest path" constraint. It has something better: **`add_circuit`**, with a
dedicated propagator that reasons about connectivity directly.

**What `add_circuit` does.** You hand it a list of arcs, each a triple `(tail, head, literal)`. It
enforces that the arcs whose literals are true form **exactly one circuit** covering the nodes —
with one escape hatch. A **self-loop** `(x, x, literal)` means "skip node `x` entirely". So the real
contract is: *one circuit through a subset of the nodes, and the self-loops say which subset.*

Two tricks turn that into a longest path:

1. **Optional nodes.** Give each number a self-loop whose literal is `used[x].negated()`. Now
   "not on the circuit" and "not in the chain" are literally the same variable, and we can maximise
   the number of `used[x]` that are true.
2. **Circuit into path.** Add a **dummy node** joined to and from every number. The single circuit
   must pass through the dummy exactly once. Cut the dummy out and what is left is a path: the arc
   `dummy -> a` names the first number of the chain, and `b -> dummy` names the last.

Node 0 is free for the dummy, since the puzzle's numbers start at 1.

**Why not the obvious encoding?** You could give each number a position variable and constrain
consecutive positions to divide (an MTZ-style formulation). It is correct and it is slow: the solver
has no idea whether a partial assignment has already stranded a group of numbers until almost
everything is decided. `add_circuit` prunes the moment a node becomes unreachable. Here that is the
difference between a tenth of a second and coming back after lunch.

In [ ]:
DUMMY = 0


def build_model():
    """Build the CP-SAT model. Returns (model, used, arc_literal) with nothing optimised yet."""
    model = cp_model.CpModel()

    # used[x] is true when x appears somewhere in the chain.
    used = {}
    for x in NUMBERS:
        used[x] = model.new_bool_var(f"used_{x}")

    arcs = []  # (tail, head, literal) triples handed to add_circuit
    arc_literal = (
        {}
    )  # (tail, head) -> literal, so the chain can be read back afterwards

    for x in NUMBERS:
        # A true self-loop is how add_circuit is told to leave this node out altogether.
        arcs.append((x, x, used[x].negated()))

        # The dummy node stands for "off the end of the chain": dummy -> x means the chain
        # starts at x, and x -> dummy means it finishes there.
        starts_here = model.new_bool_var(f"starts_{x}")
        ends_here = model.new_bool_var(f"ends_{x}")
        arcs.append((DUMMY, x, starts_here))
        arcs.append((x, DUMMY, ends_here))
        arc_literal[(DUMMY, x)] = starts_here
        arc_literal[(x, DUMMY)] = ends_here

    # Every legal link becomes two arcs, because the chain may traverse it in either direction.
    for x, y in EDGES:
        forwards = model.new_bool_var(f"arc_{x}_{y}")
        backwards = model.new_bool_var(f"arc_{y}_{x}")
        arcs.append((x, y, forwards))
        arcs.append((y, x, backwards))
        arc_literal[(x, y)] = forwards
        arc_literal[(y, x)] = backwards

    model.add_circuit(arcs)
    return model, used, arc_literal

In [ ]:
def chain_from_solution(solver, arc_literal):
    """Read a solved model's arcs back as the chain, in order, with the dummy removed."""
    follows = {}
    for (tail, head), literal in arc_literal.items():
        if solver.value(literal) == 1:
            follows[tail] = head

    chain = []
    current = follows[DUMMY]  # the arc leaving the dummy points at the first number
    while current != DUMMY:
        chain.append(current)
        current = follows[current]
    return chain


def longest_chain(add_requirements=None, verbose=False, time_limit_seconds=60):
    """The longest legal chain, as (length, chain). Length is -1 when the requirements are impossible.

    `add_requirements` is an optional function(model, used) used in section 8 to ask
    "what is the longest chain that also does X?".
    """
    model, used, arc_literal = build_model()

    if add_requirements is not None:
        add_requirements(model, used)

    numbers_used = []
    for x in NUMBERS:
        numbers_used.append(used[x])
    model.maximize(sum(numbers_used))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    solver.parameters.num_workers = 8
    status = solver.solve(model)

    if verbose:
        print("status         :", solver.status_name(status))
        print("best chain     :", solver.objective_value)
        print("proven ceiling :", solver.best_objective_bound)
        print("wall time      :", round(solver.wall_time, 2), "seconds")

    if status != cp_model.OPTIMAL and status != cp_model.FEASIBLE:
        return -1, []

    chain = chain_from_solution(solver, arc_literal)
    return len(chain), chain

## 7. Solve it

`OPTIMAL` with the best chain and the proven ceiling equal is what we are after. That pair of equal
numbers is the whole point: the first says "here is a chain this long", the second says "and there
is no longer one".

In [ ]:
best_length, best_chain = longest_chain(verbose=True)

print()
print("longest chain:", best_length, "numbers")
print(best_chain)
print("legal?", check(best_chain))

**77**, proved, in about a tenth of a second — no greedy warm start needed, and the hand-derived
ceiling of 91 was 14 too generous.

Two things worth noticing before moving on:

- There are *many* different 77-chains. Re-running this cell can print a different one; only the
  length is unique. The chain above is one witness, not "the" answer.
- The solver spent almost all of its effort on the *proof*. Finding a 77-chain is easy; showing
  that 78 is impossible is the hard half, and it is the half a hand search can never do.

## 8. Using the solver as a microscope

Having the number is not the same as understanding it. Twenty-three numbers got left out — which
ones, and why?

The technique: re-solve with an extra requirement bolted on, and see what it costs. Each of these
is a fresh model with one added constraint, which is cheap enough to do a hundred times.

In [ ]:
def longest_chain_with(must_include=(), must_exclude=()):
    """The length of the longest chain containing every number in must_include and none in must_exclude."""

    def add_requirements(model, used):
        for x in must_include:
            model.add(used[x] == 1)
        for x in must_exclude:
            model.add(used[x] == 0)

    return longest_chain(add_requirements)[0]


def longest_chain_using_exactly(group, how_many):
    """The length of the longest chain that uses exactly `how_many` of the numbers in `group`."""

    def add_requirements(model, used):
        chosen = []
        for x in group:
            chosen.append(used[x])
        model.add(sum(chosen) == how_many)

    return longest_chain(add_requirements)[0]


OPTIMUM = best_length

### 8.1 Which numbers can never be in a longest chain?

Force each number in, one at a time. If the best chain containing it is shorter than 77, that
number is excluded from *every* optimal answer. (This is 100 separate solves and takes roughly
fifteen seconds.)

In [ ]:
never_in_a_longest_chain = []
for x in NUMBERS:
    best_with_x = longest_chain_with(must_include=[x])
    if best_with_x < OPTIMUM:
        never_in_a_longest_chain.append((x, best_with_x))

for x, best_with_x in never_in_a_longest_chain:
    print(f"{x:>3}  forcing it in caps the chain at {best_with_x}")

Eighteen numbers, and they are exactly the two structures we found by hand:

- the **ten dead-end primes** of §5.1 — 53, 59, 61, 67, 71, 73, 79, 83, 89, 97;
- the **four tails** of §5.4 — 37, 41, 43, 47 together with 74, 82, 86, 94.

Note how much sharper the solver's verdict is than the hand argument. §5.1 concluded "at most one
dead-end prime survives"; the truth is that **none** does. A chain forced to contain one tops out at
75: the prime buys you a number and costs you three, because spending an end of the chain — and a
slot on the number 1 — on a dot that leads nowhere strands better-connected numbers elsewhere.

And each tail is all-or-nothing: dropping 37 also drops 74, because 74's remaining neighbours are
just 1 and 2, and neither has a slot to spare.

### 8.2 Which numbers are in *every* longest chain?

The mirror-image experiment: ban each number in turn and see whether 77 is still reachable.

In [ ]:
in_every_longest_chain = []
for x in NUMBERS:
    if longest_chain_with(must_exclude=[x]) < OPTIMUM:
        in_every_longest_chain.append(x)

print(len(in_every_longest_chain), "numbers are compulsory:")
print(in_every_longest_chain)

### 8.3 The ten numbers with a choice — and why only five of them fit

72 numbers are compulsory, 18 are impossible. That leaves ten with any freedom at all, and since
$$72 + 5 = 77$$, exactly five of the ten make it in. Which ten?

In [ ]:
def smallest_prime_factor(x):
    """The smallest prime that divides x."""
    candidate = 2
    while candidate * candidate <= x:
        if x % candidate == 0:
            return candidate
        candidate = candidate + 1
    return x


impossible_numbers = []
for x, _ in never_in_a_longest_chain:
    impossible_numbers.append(x)

FLEXIBLE_NUMBERS = []
for x in NUMBERS:
    if x not in in_every_longest_chain and x not in impossible_numbers:
        FLEXIBLE_NUMBERS.append(x)

for x in FLEXIBLE_NUMBERS:
    factor = smallest_prime_factor(x)
    print(f"{x:>3} = {factor} x {x // factor}    neighbours {NEIGHBOURS[x]}")

Every one of them is an **odd number with two odd prime factors** — 35, 39, 51, 55, 57, 65, 77, 85,
91, 95 — and their neighbourhoods are tiny: 1, their two prime factors, and (for 35 and 39 only) a
single even multiple. They are precisely the numbers fighting over the hub slots of §5.3.

So we can watch the fight directly: force the chain to use exactly $$k$$ of them, for every $$k$$.

In [ ]:
for how_many in range(0, len(FLEXIBLE_NUMBERS) + 1):
    print(
        f"using exactly {how_many:>2} of them -> longest chain {longest_chain_using_exactly(FLEXIBLE_NUMBERS, how_many)}"
    )

A clean trade-off curve peaking at five. Take fewer and you leave numbers on the table; take more
and each extra one eats two hub slots that some other part of the chain needed, so the chain gets
*shorter*. Using all ten costs eight numbers relative to the optimum.

This is the counting argument of §5.3 with the vague parts filled in. Hand reasoning gave
$$t \le 9$$ for the hub-fed set; the solver, which also accounts for the hubs' obligations to the
rest of the chain, says the effective limit is five.

### 8.4 What is the number 1 actually worth?

Here is the experiment that changed my mind about this puzzle. Since 1 divides everything, it can be
spliced into any chain anywhere, so it looks like a free +1. Take it away and see.

In [ ]:
def ban_the_number_one(model, used):
    model.add(used[1] == 0)


length_without_one, chain_without_one = longest_chain(ban_the_number_one)

# Which numbers that every 77-chain contains can this one no longer reach?
compulsory_numbers_lost = []
for x in in_every_longest_chain:
    if x not in chain_without_one:
        compulsory_numbers_lost.append(x)

print("longest chain with 1    :", OPTIMUM)
print("longest chain without 1 :", length_without_one)
print("compulsory numbers it can no longer afford:", compulsory_numbers_lost)

Not 1 — **4**, and the printout says where the other three went: **31, 62 and 93**.

Look at that little family. $$31$$ links only to $$\{1, 62, 93\}$$; $$62 = 2 \times 31$$ links only
to $$\{1, 2, 31\}$$; $$93 = 3 \times 31$$ links only to $$\{1, 3, 31\}$$. It is a three-dot island
reachable from the main body of the graph through just three doors: 1, 2 and 3. With 1 available the
chain strolls in through 1 and out through 2. Without it, visiting the island means spending a slot
at **both** 2 and 3 — and those two are the busiest hubs in the graph. The solver's verdict is that
the detour is not worth it, and it abandons all three numbers.

So 1 is not padding. It is the graph's universal joint: the one dot adjacent to everything, which
makes it the cheapest way to reach any awkward corner. That is also why spending its slots on a
dead-end prime in §8.1 is such a bad trade — 1 has two slots, and both are doing real work.

### 8.5 The price of greed

Two more one-line experiments, for the structures from §5.1 and §5.4. How does the chain length
respond as we force in more dead-end primes, or more of the tails?

In [ ]:
for how_many in range(0, 4):
    print(
        f"exactly {how_many} dead-end primes -> longest chain {longest_chain_using_exactly(DEAD_END_NUMBERS, how_many)}"
    )

print()
for how_many in range(0, 5):
    print(
        f"exactly {how_many} tail primes     -> longest chain {longest_chain_using_exactly(TAIL_PRIMES, how_many)}"
    )

A length of $$-1$$ means *infeasible*: no chain of any length exists.

The collapse is dramatic and it is exactly the hand argument from §5.1 and §5.4, now with numbers
attached. Two dead-end primes force the chain to be $$p, 1, q$$ — three numbers, the entire puzzle
thrown away. Three are impossible outright, because a path cannot have three ends. The tails behave
the same way one step further out: three of them squeeze a chain down to 8 numbers, four cannot
coexist at all.

## 9. The answer

Everything adds up:

| | count |
|---|---|
| dead-end primes 53…97, which cost more than they give | 10 |
| the tails 37/74, 41/82, 43/86, 47/94 | 8 |
| odd two-prime numbers that lose the fight for hub slots | 5 |
| **in the chain** | **77** |
| total | 100 |

The longest chain has **77 numbers**, and CP-SAT proves no 78-chain exists.

In [ ]:
def show_chain(chain):
    """Print a chain ten numbers to a line, with its length and the checker's verdict."""
    for position in range(0, len(chain), 10):
        row = chain[position : position + 10]
        print("  " + "  ".join(f"{value:>3}" for value in row))
    print()
    print("length:", len(chain))
    print("problems found by check():", check(chain))


show_chain(best_chain)

### What I take away from this one

- **Recognising the shape of a problem is most of the work.** "Chain of numbers" is a path in a
  graph; once that is said out loud, the whole toolbox opens.
- **`add_circuit` with self-loops and a dummy node** is the pattern for any "longest / best path or
  tour over an optional subset" question. Worth keeping.
- **A solved model is a lab, not a full stop.** The answer took a tenth of a second; §8 is where the
  puzzle was actually understood, and it is all one-line variations on a model that was already
  built.
- The hand analysis was not wasted: it found the right structures (§5.1 and §5.4 named 18 of the 23
  missing numbers exactly). What it could not do was resolve the competition between them. That
  division of labour — humans find the structure, the solver settles the bookkeeping — is the point
  of the whole exercise.